Performing data Augumentation.


In [ ]:
import numpy as np
import tensorflow as tf

data = np.load("/content/drive/MyDrive/Datasets/fer2013_preprocessed.npz")

x_train = data["x_train"]
y_train = data["y_train"]

Find images per class

In [ ]:
unique, counts = np.unique(y_train, return_counts=True)

print("Class Distribution:\n")
for cls, cnt in zip(unique, counts):
    print(f"Class {cls}: {cnt} samples")


Class Distribution:

Class 0: 3230 samples
Class 1: 342 samples
Class 2: 3287 samples
Class 3: 5807 samples
Class 4: 3924 samples
Class 5: 3852 samples
Class 6: 2526 samples


A method to perform augumentation by rotating, zooming, changing contrast and other modifiable details for an image.

In [ ]:
data_augumenter = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1)
])

Find minority classes

In [ ]:
x_balanced = []
y_balanced = []

max_count = np.max(counts)

for cls in unique:
  cls_images = x_train[y_train == cls]
  cls_labels = y_train[y_train == cls]

  num_to_generate = max_count - len(cls_images)

  x_balanced.append(cls_images)
  y_balanced.append(cls_labels)

  if (num_to_generate > 0):
    new_images = []

    for i in range(num_to_generate):
      img = cls_images[i % len(cls_images)]
      aug_img = data_augumenter(tf.expand_dims(img, axis=0))[0]
      new_images.append(aug_img)

    x_balanced.append(tf.stack(new_images))
    y_balanced.append(np.ones(num_to_generate, dtype=int) * cls)

x_balanced = tf.concat(x_balanced, axis=0)
y_balanced = tf.concatenate(y_balanced, axis=0)

print("Balanced dataset shape: ", x_balanced.shape, y_balanced.shape)